# mart_dev.runs_per_day

Fill in every `<...>` in the code cells below, brackets and all: the notebook won't run with
one left. They are:
  - `<SOURCE_NAME>` and `<SOURCE_TABLE>`: the table it reads, by its file in references/ and
    its name there, such as `ops__job_runs` and `job_runs`;
  - `<READS>`: which of its rows write() reads, as a condition on it, such as
    `equals(job_runs.dt, "2026-09-24")` or `equals(region_costs.region, "eu")`. On a
    partitioned table, it must filter one of its partition columns. A table that isn't
    partitioned can be read whole: delete the line `READS = <READS>` and each line
    `WHERE(READS),` instead;
  - `<GROUP_COLUMN>`: the column each row is for, such as `job_id`;
  - `<COUNT_NAME>`: the name of the count it saves, in quotes, such as `"runs"`;
  - `<PARTITION_COLUMN>`: the column of the table it reads that fills this table's partition
    column, such as `dt` or `region`. If this table isn't partitioned, delete the two lines
    of rows() naming `<PARTITION_COLUMN>`, and write `PARTITIONED_BY = []`, instead.

Then save it: `project.reference_from_logic("mart_dev.runs_per_day")`, a new_table Workflow step,
writes its Table reference, `references/mart_dev__runs_per_day.py`, from the logic, for you to answer.
It lists the columns write() saves, in the order its SELECT gives them: here
<GROUP_COLUMN>, typed as the table read types it, then <COUNT_NAME>, a count, "bigint", and
last its partition column, if it has one, as PARTITIONED_BY names it. Its Key, proposed from
the query, is the column each row is for, and the partition column if it has one, such as
key=["job_id", "region"].

One row per job and day: how many runs that job had that day, from ops.job_runs.

A Saved table worked out from a table it reads: a count per group of the rows READS picks.
It is partitioned as PARTITIONED_BY says, or not at all.

Made by `project.new_table("mart_dev.runs_per_day")` from
`admin/templates/saved_table.py`, which filled in the database, the table and its file's name,
mart_dev__runs_per_day. Run the cells top to bottom once its Table reference is written from the logic.

The code cells: 1 the Project and your query; 2 the logic, which starts `# logic`; 3 try it;
4 its lineage; 5 its Hive, in mart_dev__runs_per_day.hql; 6 its rows in the Reference sheet. The logic only
imports and defines, so `project.load("mart_dev.runs_per_day")` can run it from anywhere else, for
lineage across your tables and for `admin/run_tables.ipynb`, which sends its `write()`.

In [ ]:
# 1. The Project and your query
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # the Project folder: two up from here

from sqlglot_composer import export_hive, export_lineage, project, run_query, show_hive
from warehouse import query

In [ ]:
# logic
# 2. The logic: imports, the rows it reads, and functions only
from sqlglot_composer import (
    AS, FROM, GROUP_BY, INSERT_OVERWRITE, SELECT, WHERE, Clause, Statement, between, count_rows,
    create_table, statement,
)
from references.mart_dev__runs_per_day import runs_per_day  # this table's own Table reference
# each table it reads, such as: from references.ops__job_runs import job_runs
from references.ops__job_runs import job_runs

# The rows write() reads: change it to write others. A condition that calls something other
# than equals, such as between or is_in, needs it added to the import above.
READS = between(job_runs.dt, "2026-09-23", "2026-09-24")

# This table's partition columns, in the order its SELECT gives them, last: [] for none.
PARTITIONED_BY = ["dt"]


def create() -> Statement:
    """Step 1: create mart_dev.runs_per_day from its Table reference, if it isn't there yet."""
    return create_table(runs_per_day, may_exist=True)


def rows() -> list[Clause]:
    """The clauses preview and write share, so the two can never disagree: a count per group
    of the rows READS picks.

    A partitioned table's partition columns come last in SELECT, in its Table reference's
    order, named as it names them: INSERT_OVERWRITE fills each Partition from the rows' own
    values, and replaces only the Partitions they fill. They stay in GROUP_BY too, so each
    Partition's rows stay apart.
    """
    return [
        SELECT(
            job_runs.job_id,  # what each row is for
            AS(count_rows(), "runs"),
            job_runs.dt,  # this table's partition column, last
        ),
        FROM(job_runs),
        WHERE(READS),
        GROUP_BY(
            job_runs.job_id,
            job_runs.dt,  # kept, so each Partition's rows stay apart
        ),
    ]


def preview() -> Statement:
    """Step 2: the rows write() would save, as a SELECT to run and check first."""
    return statement(*rows())


def write() -> Statement:
    """Step 3: save the rows, replacing the Partitions they fill, or the whole table if it
    isn't partitioned: sent twice, it leaves the same rows, not twice as many."""
    return statement(INSERT_OVERWRITE(runs_per_day), *rows())


In [ ]:
# 3. Try it: the Hive of each step, then the rows
show_hive([create(), write()])
run_query(preview(), query)

In [ ]:
# 4. Lineage: a new pair in lineage/, named by the time and this table, beside the ones before
export_lineage(write())

In [ ]:
# 5. Its Hive: mart_dev__runs_per_day.hql beside this notebook, from the logic as last saved
export_hive()

In [ ]:
# 6. The Reference sheet: this table's rows, from references/mart_dev__runs_per_day.py
project.refresh_reference_sheet("mart_dev.runs_per_day")